#### Read a folder of documents and answer questions

In [ ]:
import os
from huggingface_hub import InferenceClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LangChain for orchestration, Chroma for storage, and HuggingFace to run the Llama 3 model on HF's services of Inference API.

In [6]:
# Loading the PDF document

def load_documents(folder_path: str):
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder '{folder_path}' does not exist")

    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            print(f"📄 Loading: {filename}")
            try:
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            except Exception as e:
                print(f"❌ Error loading {filename}: {e}")
    return documents

# Chunking the documents caause the LLMs have a context window limit, so we need to split the documents into smaller chunks
def split_text(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200, # creates a sliding window, making sure we don’t lose context if a sentence is split between chunks
    )
    chunks = splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

In [ ]:
# Embeddings - we need to turn the text chunks into lists of numbers
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2" # lightweight, open-source model that runs quickly on CPU, the this model is only ~80MB
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2958.47it/s]


In [8]:
# Remote LLM on HF
client = InferenceClient(token=os.getenv("HF_TOKEN"))
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
# Storing the embeddings in a vector store - we can use Chroma to store the embeddings in a vector database, which allows us to perform similarity searches on the embeddings
def create_vector_store(chunks):
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory="../data/train/rag/chroma_db", # saves the database in a folder called ./chroma_db. That way, you don’t have to rebuild the database every time you restart the app; it stays saved.
        collection_name="rag_docs"
    )
    return vector_store

In [10]:
def query_rag_system(query_text, vector_store):
    """
    Links the user, the database, and the LLM : looks at the users question and finds the top 3 most relevant chunks (k=3). Then, it puts those chunks into a strict prompt:
    """
    # Retrieve top 3 relevant chunks
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(query_text)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Generate answer via HF Inference API
    result = client.chat_completion(
        messages=[{
            "role": "user",
            "content": f"""You are a helpful assistant.
            Answer ONLY using the context below.
            If the answer is not present, say "I don't know."

            Context:
            {context}

            Question:
            {query_text}"""
                    }],
                    model=MODEL,
                    max_tokens=300
                )

    return result.choices[0].message.content.strip()


In [13]:
# Run the Pipeline

folder_path = "../data/raw/pdf"  # Path to the folder containing PDF files


if not os.path.exists("../data/train/rag/chroma_db"):
    print("No vector DB found. Creating one...")
    docs = load_documents(folder_path)
    chunks = split_text(docs)
    vector_store = create_vector_store(chunks)
    print("Vector database created")
else:
    print("Loading existing vector DB...")
    vector_store = Chroma(
        persist_directory="../data/train/rag/chroma_db",
        embedding_function=embedding_function,
        collection_name="rag_docs"
    )

while True:
    query = input("\nAsk a question (or type 'exit'): ")
    if query.lower() == "exit":
        break

    print("Thinking...")
    answer = query_rag_system(query, vector_store)
    print("\n Answer:\n", answer)

No vector DB found. Creating one...
📄 Loading: Full-49.pdf
📄 Loading: Full-47.pdf
📄 Loading: Full-48.pdf
Created 7 chunks
Vector database created
Thinking...

 Answer:
 According to the context, the Moon is mentioned several times, but there is no direct definition or description of what the Moon is. However, it is described as being in a "meek shine" and having its shadow stolen by the Earth in a lunar eclipse. If you're looking for more information, I can try to help you find it!
Thinking...

 Answer:
 I'm here to help! Since I only have the context provided, I'll do my best to answer your question. If I don't know the answer, I'll say "I don't know."

Please go ahead and ask your question, and I'll do my best to assist you!
